# Fase 2 — Auditoría DENUE

**Proyecto:** CRM-Granos-MX  
**Fecha:** 2026-04-30

Este notebook ejecuta los chequeos de calidad definidos en [`src/audit/`](../src/audit/) sobre la tabla `establecimientos` y muestra el reporte en formato visual.

Mientras la tabla esté vacía (pre-Fase 3), el reporte se ve casi todo en cero. Tras la primera descarga DENUE, este notebook empezará a tener señales reales.

## Qué hace cada celda

1. Conecta a la DB local.
2. Corre `run_all()` y obtiene el `ReporteAuditoria`.
3. Imprime el resumen.
4. Lista errors y warnings con sus ejemplos.
5. Visualización: tabla por severidad.

Para detalle de cada chequeo y su política de severidad, ver [`docs/auditoria/criterios_calidad.md`](../docs/auditoria/criterios_calidad.md).

In [ ]:
import sys
from pathlib import Path

# Asegura que el paquete `src` esté importable desde el notebook
sys.path.insert(0, str(Path.cwd().parent))

from src.audit import run_all
from src.core.db import SessionLocal

with SessionLocal() as db:
    reporte = run_all(db, universo='establecimientos:full')

print('Resumen del reporte:')
for k, v in reporte.resumen().items():
    print(f'  {k}: {v}')

In [ ]:
# Errors (severidad='error', total_problemas>0)
if reporte.errores:
    print(f'Hay {len(reporte.errores)} chequeos con error:\n')
    for h in reporte.errores:
        print(f'  {h.chequeo:<40} {h.total_problemas:>5} / {h.total_evaluados:>5}  ({h.pct_problemas}%)')
        for ej in h.ejemplos[:3]:
            print(f'      ej: {ej}')
else:
    print('Sin errors. ✓')

In [ ]:
# Warnings
if reporte.warnings:
    print(f'{len(reporte.warnings)} chequeos con warning:\n')
    for h in reporte.warnings:
        print(f'  {h.chequeo:<40} {h.total_problemas:>5} / {h.total_evaluados:>5}  ({h.pct_problemas}%)')
else:
    print('Sin warnings. ✓')

In [ ]:
# Visualización tabular completa con pandas
import pandas as pd

df = pd.DataFrame([
    {
        'chequeo': h.chequeo,
        'severidad': h.severidad,
        'evaluados': h.total_evaluados,
        'problemas': h.total_problemas,
        'pct': h.pct_problemas,
    }
    for h in reporte.hallazgos
])
df.style.background_gradient(subset=['pct'], cmap='Reds')

## Próximos pasos al ver señales reales

Cuando este notebook empiece a mostrar números reales tras Fase 3:

- **Errors > 0** → revisar la lógica de carga DENUE antes de seguir.
- **% NULL alto en `telefono`** → priorizar enriquecimiento Google Places (Fase 4).
- **Warnings de fuzzy duplicates** → fusionar registros antes de scoring.
- **Distribución por SCIAN desbalanceada** → revisar si la descarga cubrió todos los SCIAN objetivo.

El reporte se puede serializar a JSON con `reporte.model_dump_json()` y guardarlo en `data/processed/reporte_auditoria_<fecha>.json` para tener histórico.